# Black Box Optimization.

Import libraries:

In [ ]:
import numpy as np
from numpy.linalg import norm
import plotly.graph_objects as go
from scipy.optimize import brent
from scipy.special import gamma
from scipy.stats import multivariate_normal
from scipy.linalg import fractional_matrix_power
import math
from copy import copy

Set random seed for reproducability of code in stochastic methods:

In [ ]:
np.random.seed(1234567890)

## Ackley Function.

In [ ]:
def ackley(point):
    """
        2D Ben Ackley.
        Global minimum: f(0,0) = 0 in 2D.
    """
    x = point[0]
    y = point[1]
    A=20
    B=0.2
    return -A*np.exp(-B*np.sqrt((x**2+y**2)/2))-np.exp((np.cos(math.pi*2*x)+np.cos(math.pi*2*y))/2)+A+np.e


## Cross in tray function.

In [ ]:
def cross_in_tray(point):
    """
    2D Cross in tray function.
    Global minimum: f(x,y)=-2.06261 in 2D.
    for (x,y): (1.3491, −1.3491), (1.3491, 1.3491), (−1.3491, 1.3491) and (−1.3491, −1.3491).
    """
    x = point[0]
    y = point[1]
    return -0.0001*np.pow((np.abs(np.sin(x)*np.sin(y)*np.exp(np.abs(100-np.sqrt(x**2+y**2)/math.pi)))+1),0.1)

## Eggholder function.

Defined for [-512,512] in both $x$ and $y$.
Set default high value of $10^{10}$ outside definition bounds to prevent optimization algorithms for exploring outside bounded area.

In [ ]:
def eggholder(point):
    """
    2D Eggholder function.
    Global minimum: f(512,404.2319)=-959.6407 in 2D.
    Defined for [-512,512] in x and y.
    """
    x = np.asarray(point[0], dtype=float)
    y = np.asarray(point[1], dtype=float)

    z = (-(y + 47) * np.sin(np.sqrt(np.abs(y + x / 2 + 47)))
         - x * np.sin(np.sqrt(np.abs(x - (y + 47)))))

    out_of_bounds = (np.abs(x) > 512) | (np.abs(y) > 512)
    return np.where(out_of_bounds, 10**10, z)

------------------------------------

## Methods

Methods implemented:
- Coordinate/Compass Search.
- Fixed Step Cyclic Coordinate Search.
- Line Search Cyclic Coordinate Search.
- Powells Method.
- Hooke-Jeeves.
- Generalized Pattern Search.
- Mesh Adaptive Direct Search.
- Divided Rectangles.
- Covariance Matrix Adaption Evolutionary Strategy.
- Nelder-Mead Simplex Method.

--------------------

## Helper functions.

Some basic helper functions used in parts of the code, and visualizing results.

In [ ]:
def plot_function(function, pointcosts=None, start_x=-10, end_x=10, start_y=-10, end_y=10, resolution=300):
    x = np.linspace(start_x, end_x, resolution)
    y = np.linspace(start_y, end_y, resolution)
    X, Y = np.meshgrid(x, y)
    Z = function((X, Y))

    #Default to no point unless passed.
    if pointcosts is None:
        pointcosts = []

    #Get plotted points.
    px, py = (pointcosts[:, 0], pointcosts[:, 1])
    pz = pointcosts[:, 2]

    fig = go.Figure()
    #Add 2d top-down plot.
    fig.add_trace(go.Heatmap(x=x, y=y, z=Z, colorscale="Viridis", colorbar=dict(title="Cost value")))
    fig.add_trace(go.Scatter(x=px, y=py, mode='lines+markers', name='Optimized path'))

    #Add 3d plot.
    fig.add_trace(go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", showscale=False, visible=False))
    fig.add_trace(go.Scatter3d(x=px, y=py, z=pz, mode='lines+markers', name='Optimized path', visible=False))

    #Add figure layout with 2d/3d button switch.
    fig.update_layout(
        title=f"{function.__name__.replace("_", " ").title()} Function.",
        width=600, height=600,
        xaxis=dict(title='X'),
        yaxis=dict(title='Y', scaleanchor='x', scaleratio=1),
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Cost',
                    domain=dict(x=[0, 0], y=[0, 0])),  # collapsed until 3D chosen
        updatemenus=[dict(
            type="buttons", direction="left", x=0.5, y=1.15, xanchor="center",
            buttons=[
                dict(label="2D", method="update",
                     args=[{"visible": [True, True, False, False]},
                           {"xaxis.visible": True, "yaxis.visible": True,
                            "scene.domain.x": [0, 0], "scene.domain.y": [0, 0]}]),
                dict(label="3D", method="update",
                     args=[{"visible": [False, False, True, True]},
                           {"xaxis.visible": False, "yaxis.visible": False,
                            "scene.domain.x": [0, 1], "scene.domain.y": [0, 1]}]),
            ]
        )]
    )
    fig.show()

In [ ]:
def line_search(function, point, direction):
    point = np.array([point[0],point[1]])
    direction = np.array([direction[0],direction[1]])
    def phi(t):
        x = point[0] + t * direction[0]
        y = point[1] + t * direction[1]
        return function((x, y))
    t = brent(phi)
    x = point[0] + t * direction[0]
    y = point[1] + t * direction[1]
    return (x,y)

In [ ]:
def positive_spanning_set(directions):
    directions = np.array(directions)
    D = np.array(directions, dtype=float)
    m, n = D.shape
    if m < n + 1:
        return False
    # Quick necessary check: directions must not all lean the same way
    # on ANY single axis-aligned test — catches most real failures fast.
    return np.linalg.matrix_rank(D) == n and np.any(np.sum(np.sign(D), axis=0) == 0)

In [ ]:
def complete_poll(function, point, directions, step_size):
    new_point = point
    new_cost = function(point)

    for dir in directions:
        candidate_point = (point[0]+step_size*dir[0], point[1]+step_size*dir[1])
        candidate_cost = function(candidate_point)
        if candidate_cost<new_cost:
            new_point = candidate_point
            new_cost = candidate_cost
        
    return (new_point, new_cost)

--------------------------------

## Coordinate search ( Compass Search ).

In [ ]:
def coordinate_search(function, x, y, iterations=100, tolerance=1e-5, step_size=0.1, step_decay=0.9):
    """
    Minimizes a 2D function via coordinate/compass search.
    Polls opportunistically along each coordinate axis.
    Decreases step size by stepdecay factor each time no improvement was found.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum step_size allowed before giving up.
        - step_size, start step size in each coordinate axis.
        - step_decay, factor to decrease step_size by whenever no improving point was found.

    Returns:
        Numpy array of (x,y,cost) points.
    """
    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need positive step size.
    if step_size <= 0:
        raise ValueError("Step size must be positive.")

    #Need (0,1) for step decay.
    if not (0<step_decay<1):
        raise ValueError("Step decay must be between 0 and 1.")

    directions = [(0,1), (0,-1), (1,0), (-1,0)]
    points = [(x,y)]
    costs = [function((x,y))]

    for i in range(iterations):
        #Last point found.
        prev_point = points[-1]
        prev_cost = costs[-1]

        #Store if any direction improved.
        improved = False

        #Loop if no improvement has been found.
        for dir in directions:
            candidate_point = (prev_point[0]+step_size*dir[0],prev_point[1]+step_size*dir[1])
            candidate_cost = function(candidate_point)

            #If cost is better than best found candidate cost so far, or previous cost.
            if candidate_cost < prev_cost:
                improved=True
                points.append(candidate_point)
                costs.append(candidate_cost)
                break

        #If no improvement found.
        if not improved:
            step_size*=step_decay

            #If step size falls below tolerance, quit.
            if step_size < tolerance:
                break

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, coordinate_search(cross_in_tray,3,6,100))
plot_function(eggholder, coordinate_search(eggholder,4,50,100,0.005,3,0.9), start_x=-100, end_x=100, start_y=-100, end_y=100)
plot_function(ackley, coordinate_search(ackley,-8,5,100,0.005,1,0.9))

## Fixed Step Cyclic Coordinate Search.

In [ ]:
def cyclic_search(function, x, y, iterations=100, tolerance=1e-5, step_size=1):
    """
    Minimizes a 2D function using fixed step size cyclic coordinate search.
    Polls opportunistically one axis at a time.
    Does not decrease step_size on improvement failure - fixed step size.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum cost difference between previous point and current point allowed before giving up.
        - step_size, start step size in each coordinate axis.

    Returns:
        Numpy array of (x,y,cost) points.
    """
    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need non-negative tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need non-zero positive step size.
    if step_size <= 0:
        raise ValueError("Step size must be positive.")

    x_directions = [(-1,0),(1,0)]
    y_directions = [(0,-1),(0,1)]
    x_axis = True

    points = [(x, y)]
    costs = [function((x,y))]

    for i in range(iterations):
        #Last point found.
        prev_point = points[-1]
        prev_cost = costs[-1]

        #Find what axis to search.
        directions = (x_directions if x_axis else y_directions)

        #Go through axis points.
        for dir in directions:
            candidate_point = (prev_point[0]+step_size*dir[0],prev_point[1]+step_size*dir[1])
            candidate_cost = function(candidate_point)

            #If cost is better than best found candidate cost, or previous cost.
            if candidate_cost < prev_cost:
                points.append(candidate_point)
                costs.append(candidate_cost)
                break
        
        x_axis = not x_axis

        #Quit if cost tolerance is hit.
        if len(costs)>=2 and np.abs(costs[-1]-costs[-2])<tolerance:
            break

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, cyclic_search(cross_in_tray,3,6))
plot_function(eggholder, cyclic_search(eggholder,3,69),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, cyclic_search(ackley,-8,5))

## Line Search Cyclical Coordinate Search.


In [ ]:
def line_search_cyclic_search(function, x, y, iterations=100, tolerance=1e-5):
    """
    Minimizes a 2D function using line search cyclic coordinate search.
    Polls opportunistically one axis at a time using a line search.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum cost difference between previous point and current point allowed before giving up.

    Returns:
        Numpy array of (x,y,cost) points.
    """
    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need non-negative tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    x_directions = [(1,0)]
    y_directions = [(0,1)]
    x_axis = True

    points = [(x, y)]
    costs = [function((x,y))]

    for i in range(iterations):
        #Last point found.
        prev_point = points[-1]
        prev_cost = costs[-1]

        #Find what axis to search.
        directions = (x_directions if x_axis else y_directions)

        #Go through axis points.
        for dir in directions:
            candidate_point = line_search(function, prev_point, dir)
            candidate_cost = function(candidate_point)

            #If cost is better than best found candidate cost, or previous cost.
            if candidate_cost < prev_cost:
                points.append(candidate_point)
                costs.append(candidate_cost)
                break
        
        x_axis = not x_axis

        #Quit if cost tolerance is hit.
        if len(costs)>=2 and np.abs(costs[-1]-costs[-2])<tolerance:
            break

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, line_search_cyclic_search(cross_in_tray,2,4))
plot_function(eggholder, line_search_cyclic_search(eggholder,60,0),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, line_search_cyclic_search(ackley,-5,5))

## Powells Method.

In [ ]:
def powells_method(function, x, y, iterations = 100, tolerance=1e-5):
    """
    Minimizes a 2D function using Powells Method.
    Line searches and moves along each direction.
    Then line search along net movement direction found.
    Replace highest decreasing movement direction with new net direction.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum cost difference between previous point and current point allowed before giving up.

    Returns:
        Numpy array of (x,y,cost) points.
    """

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    directions = [(1,0),(0,1)]
    points = [(x,y)]
    costs = [function((x,y))]

    for i in range(iterations):
        prev_point = points[-1]
    
        #Line search and go to new point for each dimension.
        candidate_points = [prev_point]
        for ii in range(len(directions)):
            candidate_point = line_search(function, candidate_points[-1], directions[ii])
            candidate_points.append(candidate_point)
        
        #Remove the largest decrease direction.
        reductions = np.array([function(candidate_points[i])-function(candidate_points[i+1]) for i in range(len(directions))])
        directions.pop(int(np.argmax(reductions)))
        
        #Append net line direction.
        new_direction = (
            candidate_points[-1][0]-candidate_points[0][0],
            candidate_points[-1][1]-candidate_points[0][1]
        )
        directions.append(new_direction)

        #Search net line direction.
        new_point = line_search(function,candidate_points[-1],directions[-1])
        new_cost = function(new_point)

        #Update lists.
        points.append(new_point)
        costs.append(new_cost)

        #Early stopping due to finding minimum.
        if np.abs(costs[-2]-new_cost)<=tolerance:
            break

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, powells_method(cross_in_tray,0.1,-3.1,100))
plot_function(eggholder, powells_method(eggholder,-78,82,100),start_x=-200,end_x=200,start_y=-200,end_y=200)
plot_function(ackley, powells_method(ackley,-6,3,100))

## Hooke Jeeves.

In [ ]:
def hooke_jeeves(function, x, y, iterations=100, tolerance=1e-5, step_size=1, step_decay=0.5):
    """
    Minimizes a 2D function using Hooke-Jeeves.
    Hooke-Jeeves consists of an exploratory phase and a pattern phase.
    Exploratory: complete poll around current position, denoting best direction.
    Pattern move: move twice distance in best direction, complete poll around.
    Stores best point found in exploratory and pattern, for each iteration.
    Decay step size if no improvement is found.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum step_size allowed before giving up.
        - step_size, start step size in each coordinate axis.
        - step_decay, factor to decrease step_size by whenever no improving point was found.

    Returns:
        Numpy array of (x,y,cost) points.
    """

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need (0,1) step decay.
    if step_decay < 0 or step_decay > 1:
        raise ValueError("Step decay must be between 0 and 1 for convergence.")

    #Need non-zero positive step size.
    if step_size <= 0:
        raise ValueError("Step size must be positive.")

    directions = [(-1,0),(1,0),(0,1),(0,-1)]
    points = [(x,y)]
    costs = [function((x,y))]

    for i in range(iterations):
        prev_point = points[-1]
        
        #Explore phase.
        base_point, base_cost = complete_poll(function, prev_point, directions,step_size)

        #No improvement.
        if base_point == prev_point:
            step_size*=step_decay
            if step_size <= tolerance:
                break
            continue
        
        #Pattern move.
        pattern_direction = (base_point[0]-prev_point[0], base_point[1]-prev_point[1])
        pattern_point = (base_point[0]+pattern_direction[0], base_point[1]+pattern_direction[1])

        #Probe around the pattern move point.
        around_pattern, around_cost = complete_poll(function, pattern_point, directions, step_size)

        #No need to check if around is better than base, due to base folding into polling set.
        points.append(around_pattern)
        costs.append(around_cost)

        #Update decay if pattern move failed.
        if around_cost >= base_cost:
            step_size *= step_decay
            if step_size <= tolerance:
                break
    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])


In [ ]:
plot_function(cross_in_tray, hooke_jeeves(cross_in_tray,3,6))
plot_function(eggholder, hooke_jeeves(eggholder,3,69),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, hooke_jeeves(ackley,-8,5))

## Generalized pattern search

In [ ]:
def generalized_pattern_search(function, x, y, iterations=100, tolerance=1e-5, step_size=1, step_decay=0.5, step_expansion=2, directions=None):
    """
    Minimizes a 2D function using Generalized Pattern Search.
    Generalized Pattern Search takes in a positive spanning directions set.
    Polls opportunistically along directions.
    Increases step size by step expansion factor if improvement is found.
    Decreases step size by step decay factor if no improvement is found.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum step_size allowed before giving up.
        - step_size, start step size in each direction.
        - step_decay, factor to decrease step_size by whenever no improving point was found.
        - step_expansion, factor to increase step_size by whenever an improving point was found.
        - directions, set of directions to use, must form a positive spanning set.
    Returns:
        Numpy array of (x,y,cost) points.
    """
    #Check if directions exist.
    if directions is None:
        directions = [(0,1), (0,-1), (1,0), (-1,0)]

    #Check if directions form positive spanning set.
    if not positive_spanning_set(directions):
        raise ValueError("Directions do not form positive spanning set.")

    #Confirm step decay, is actually decaying.
    if step_decay < 0 or step_decay > 1:
        raise ValueError("Step decay must be between 0 and 1 for convergence.")

    #Need non-zero positive step size.
    if step_size<=0:
        raise ValueError("Step size must be positive.")

    #Confirm step expansion actually expands.
    if step_expansion < 1:
        raise ValueError("Step expansion must be above 1 for convergence.")

    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")


    #Store points and costs.
    points = [(x,y)]
    costs = [function((x,y))]

    #Go through all iterations.
    for i in range(iterations):
        #Last point found.
        prev_point = points[-1]
        prev_cost = costs[-1]
        
        #Store if any direction improved.
        improved = False

        #Check directions.
        for j, dir in enumerate(directions):
            candidate_point = (prev_point[0]+step_size*dir[0],prev_point[1]+step_size*dir[1])
            candidate_cost = function(candidate_point)

            #Opportunistic poll.
            if candidate_cost < (prev_cost):
                step_size*=step_expansion # Step size expansion.
                points.append(candidate_point)
                costs.append(candidate_cost)
                directions.insert(0, directions.pop(j)) #Prioritize this direction next.
                improved=True
                break
        
        #If no improvement found.
        if not improved:
            step_size*=step_decay
            
            #If minimum step size tolerance hit, quit.
            if step_size < tolerance:
                break
    
    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, generalized_pattern_search(cross_in_tray,3,6,100))
plot_function(eggholder, generalized_pattern_search(eggholder,-50,95,100,1e-5,3),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, generalized_pattern_search(ackley,-8,5,100,1e-5,1))

## Nelder-Mead Simplex Method

In [ ]:
def nelder_mead(function, x, y, simplex_size=1, reflection_factor=1, expansion_factor=2, contraction_factor=0.5, shrink_factor=0.5, iterations=100, tolerance=1e-5):
    """
    Minimizes a 2D function using Nelder-Mead Simplex Method.
    Nelder-Mead tries to move a simplex by reflection, expansion, contraction or shrinkage, each iteration.

    Each iteration the worst vertex is reflected over the centroid of the others, then:
        - Reflection: reflected point beats the second worst but not the best. Replace the worst vertex with it.
        - Expansion: reflected point beats the best. Push further along that direction, keeping whichever of the expanded or reflected point is better.
        - Contraction: reflected point is no better than the second worst. Try a point between the centroid and the worst (or reflected) vertex.
        - Shrinkage: contraction fails to improve. Move every vertex toward the best one.

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum standard deviation of costs allowed before giving up.
        - simplex size, equilateral triangle simplex side length.
        - reflection factor, factor to reflect centroid over by.
        - expansion factor, factor to expand simplex by.
        - contraction factor, factor to contract simplex by.
        - shrink factor, factor to shrink simplex by.
    Returns:
        Numpy array of (x,y,cost) points.
    """

    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Need positive simplex start size.
    if simplex_size<=0:
        raise ValueError("Simplex start size must be positive valued.")

    #Need non-zero positive reflection_factor.
    if reflection_factor<=0:
        raise ValueError("Reflection factor must be positive.")

    #Need non-zero positive iterations.
    if expansion_factor<=0:
        raise ValueError("Expansion factor must be positive.")

    #Need non-zero positive contraction factor..
    if contraction_factor<=0:
        raise ValueError("Contraction factor must be positive.")

    #Need non-zero shrink in (0,1).
    if not (0 < shrink_factor < 1):
        raise ValueError("Shrink factor must be in (0,1).")

    #Optimization path.
    points = [(x,y)]
    costs = [function((x,y))]

    #Instantiate simplex around x,y.
    simplex_points = np.array([(x+simplex_size*math.cos(math.radians(90+120*i)),y+simplex_size*math.sin(math.radians(90+120*i))) for i in range(3)])
    simplex_costs = np.array([function(point) for point in simplex_points])

    for i in range(iterations):
        #Sort simplex points by cost, ascending.
        sorted_idx = np.argsort(simplex_costs)
        simplex_points = simplex_points[sorted_idx]
        simplex_costs = simplex_costs[sorted_idx]

        #Define centroid and reflection point.
        centroid = np.mean(simplex_points[:-1], axis=0)
        reflection = centroid + reflection_factor*(centroid-simplex_points[-1])
        reflection_cost = function(reflection)

        #Expansion.
        if reflection_cost < simplex_costs[0]:
            expansion = centroid+expansion_factor*(reflection-centroid)
            expansion_cost = function(expansion)

            #Expand only if expansion point is better than reflection.
            simplex_points[-1] = expansion if expansion_cost < reflection_cost else reflection 
            simplex_costs[-1] = min(expansion_cost,reflection_cost)
        #Contraction.
        elif reflection_cost>=simplex_costs[-2]:
            #Set reflection as best if better than prev best cost.
            if reflection_cost < simplex_costs[-1]:
                simplex_points[-1]=reflection
                simplex_costs[-1]=reflection_cost
            
            contraction = centroid + contraction_factor*(simplex_points[-1]-centroid)
            contraction_cost = function(contraction)
            
            #Shrink if contraction is not better.
            if contraction_cost>simplex_costs[-1]:
                for i in range(3):
                    simplex_points[i]=shrink_factor*(simplex_points[i]+simplex_points[0])
                    simplex_costs[i]=function(simplex_points[i])
            #Contract.
            else:
                simplex_points[-1]=contraction
                simplex_costs[-1]=contraction_cost
        #Reflection if nothing else beats.
        else:
            simplex_points[-1]=reflection
            simplex_costs[-1]=reflection_cost

        #Store best point.
        costs.append(simplex_costs[0])
        points.append(simplex_points[0])

        #Break if cost std. is lower than tolerance.
        if np.std(simplex_costs)<tolerance:
            break

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, nelder_mead(cross_in_tray,3,6))
plot_function(eggholder, nelder_mead(eggholder,4,50),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, nelder_mead(ackley,-8,5,simplex_size=3,reflection_factor=1.5,expansion_factor=2))

## DIvided RECTangles (DIRECT)

In [ ]:
class DirectRectangle:
    """Simple class for a rectangle in the DIRECT algorithm."""
    def __init__(self, center, cost, dimension_divisions, radius):
        self.center=center
        self.cost=cost
        self.dimension_divisions=dimension_divisions
        self.radius=radius

In [ ]:
def is_ccw(a:DirectRectangle,b:DirectRectangle,c:DirectRectangle):
    """Check if 3 Direct Rectangles are in Counter-Clockwise form"""
    return a.radius*(b.cost-c.cost)-a.cost*(b.radius-c.radius)+(b.radius*c.cost-b.cost*c.radius) < 1e-6 

In [ ]:
def find_optimal_split_cells(rectangles, tolerance):
    """
        Find best cells/DirectRectangles to split using convex hull.
    """
    convex_hull = []
    rectangles = sorted(rectangles, key=lambda rect: (rect.radius, rect.cost)) 

    #Create Convex-Hull.
    for rectangle in rectangles:
        if len(convex_hull) >= 1 and rectangle.radius == convex_hull[-1].radius:
            continue
        if len(convex_hull) >=1 and rectangle.cost <= convex_hull[-1].cost:
            convex_hull.pop()
        if len(convex_hull) >= 2 and is_ccw(convex_hull[-2], convex_hull[-1], rectangle):
            convex_hull.pop()
        convex_hull.append(rectangle)

    #Ensure cells follow tolerance criteria.
    convex_hull = [rect for rect in convex_hull if rect.radius >= tolerance]
    return convex_hull

In [ ]:
def split_cell(rectangle, scaled_function):
    """Split cell/DirectRectangle along longest side."""

    #Split information.
    center = rectangle.center
    divs_min = min(rectangle.dimension_divisions)
    divs = np.asarray(copy(rectangle.dimension_divisions))
    dirs = [i for i, div in enumerate(divs) if div == divs_min]
    delta = 3.0**(-divs_min-1)

    #Left and right cells information after split.
    centers = [(center+delta*np.eye(2)[i],center-delta*np.eye(2)[i]) for i in dirs]
    costs = [(scaled_function(C[0]), scaled_function(C[1])) for C in centers]
    min_vals = [min(y[0],y[1]) for y in costs]

    #Perform splitting.
    rectangles = []
    for i in np.argsort(min_vals):
        divs[dirs[i]]+=1
        center_pair = centers[i]
        cost_pair = costs[i]
        radius = norm(0.5*3.0**(-divs))
        rectangles.append(DirectRectangle(center_pair[0],cost_pair[0], copy(divs), radius))
        rectangles.append(DirectRectangle(center_pair[1],cost_pair[1], copy(divs), radius))

    #Middle cell after split.
    radius = norm(0.5*3.0**(-divs))
    rectangles.append(DirectRectangle(center, rectangle.cost, divs, radius))

    return rectangles


In [ ]:
def DIRECT(function, start_x, end_x, start_y, end_y, tolerance=1e-5, iterations=100):
    """
    Minimizes a 2D function using Dividing Rectangles.
    Dividing Rectangles splits the bounds into rectangles and evaluates the function at each center.
    Each iteration, it divides the rectangles on the lower convex hull of (size, cost).
    This balances searching low-cost areas with exploring large unsearched ones.
    Selected rectangles are split into thirds along their longest sides.

    Args:
        - function, function to optimize.
        - start_x, start_y, start bounds to search in.
        - end_x, end_y, end of bounds to search in.
        - iterations, max iterations before giving up.
        - tolerance, minimum cell radius allowed before giving up.
    Returns:
        Numpy array of (x,y,cost) points.
    """
    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Normalize function to [0,1]^2.
    scaled_function = lambda p: function((start_x + p[0] * (end_x - start_x), start_y + p[1] * (end_y - start_y)))

    #Original hypercube center.
    c = np.asarray([0.5,0.5])

    #Rectangles.
    rectangles = [DirectRectangle(c,scaled_function(c), [0,0], 2**(-0.5))]

    #Best found centerpoint.
    best_rectangle = rectangles[0]

    for i in range(iterations):
        #Find optimal cells to split using convex hull.
        rectangles_split = find_optimal_split_cells(rectangles, tolerance)
        
        #Remove repeated cells in split versus non-split (to not store same cell multiple times).
        rectangles[:] = [r for r in rectangles if r not in rectangles_split]

        #Split optimal cells.
        for rectangle in rectangles_split:
            rectangles.extend(split_cell(rectangle,scaled_function))

        #Store best rectangle found so far.
        best_rectangle = rectangles[np.argmin([rectangle.cost for rectangle in rectangles])]

    #Return best scoring rectangle center.
    return np.array([[best_rectangle.center[0]*(end_x - start_x) + start_x,
                  best_rectangle.center[1]*(end_y - start_y) + start_y,
                  best_rectangle.cost]])

In [ ]:
plot_function(cross_in_tray, DIRECT(cross_in_tray,-10, 10, -10, 10))
plot_function(eggholder, DIRECT(eggholder,-200, 200, -200, 200),start_x=-200,end_x=200,start_y=-200,end_y=200)
plot_function(ackley, DIRECT(ackley,-10, 10, -10, 10))

## Mesh Adaptive Direct Search (MADS)

In [ ]:
def generate_random_directions(mesh_index):
    """
        Generate random 2D directions that's a positive spanning set.
    """

    m = 2**mesh_index
    alpha = np.random.choice([-1, 1]) * m
    gamma = np.random.choice([-1, 1]) * m
    beta  = np.random.randint(-m + 1, m)
    L = np.array([[alpha, 0],
                  [beta, gamma]])
    Z = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)]
    directions = [tuple(L @ np.array(z)) for z in Z]
    max_norm = max(np.linalg.norm(d) for d in directions)
    return directions, max_norm

In [ ]:
def MADS(function, x, y, iterations=100, tolerance=1e-5):
    """
    Minimizes a 2D function using Lower Triangular Mesh Adaptive Direct Search.
    LTMADS opportunistically polls points on a mesh around the best point, in randomly chosen directions.
    If improvement point found, mesh gets coarsened. If not, mesh gets redefined. 

    Args:
        - function, function to optimize.
        - x, y, start point in optimization path.
        - iterations, max iterations before giving up.
        - tolerance, minimum poll size allowed before giving up.
    Returns:
        Numpy array of (x,y,cost) points.
    """

    points = [(x,y)]
    costs = [function((x,y))]

    #Store mesh size exponent.
    mesh_index = 0

    for i in range(iterations):
        prev_point = points[-1]
        prev_cost = costs[-1]

        #Generate mesh.
        directions, max_norm = generate_random_directions(mesh_index)
        
        #Mesh and poll information.
        mesh_size = 4**(-mesh_index)
        poll_size = mesh_size*max_norm #-> 2^(-mesh_index).

        #Break if poll size goes under tolerance.
        if poll_size < tolerance:
            break

        improved = False

        #Poll opportunistically.
        for dir in directions:
            poll_point = (prev_point[0]+mesh_size*dir[0],prev_point[1]+mesh_size*dir[1])
            poll_cost = function(poll_point)
            if poll_cost < prev_cost:
                improved=True
                points.append(poll_point)
                costs.append(poll_cost)
                break
        
        #Coarsen / redefine mesh.
        if improved:
            mesh_index=max(0, mesh_index - 1)
        else:
            mesh_index+=1

    return np.array([(x, y, cost) for (x, y), cost in zip(points, costs)])

In [ ]:
plot_function(cross_in_tray, MADS(cross_in_tray,3,6))
plot_function(eggholder, MADS(eggholder,4,50),start_x=-100,end_x=100,start_y=-100,end_y=100)
plot_function(ackley, MADS(ackley,-8,5))

## Covariance Matrix Adaptive Evolutionary Strategy.

In [ ]:
def cmaes_population_size(n):
    return 4 + math.floor(3*np.log(n))

In [ ]:
def cmaes_weights(population_size, elite_size):
    if elite_size>=population_size:
        raise ValueError("Elite size must be smaller than sample size.")

    weights = np.log(elite_size+1)-np.log(np.arange(1,elite_size+1))
    weights = weights/weights.sum()
    return weights
    

In [ ]:
def CMAES(function, x, y, iterations=100, tolerance=1e-5, sigma=0.3, population_size=None):
    """
        Minimizes a 2D function using Covariance Matrix Adaptation Evolutionary Strategy (w/ Hansen Decay and Weights).

        Each iteration, it samples a population from a Gaussian centered on the mean and keeps the best half as elites.
        The new mean is a weighted average of the elite points.
        Two exponential running averages track the trend of the mean shifts.
        One updates the covariance, stretching the distribution along directions that keep succeeding, together with the spread of the elites.
        The other, with the covariance removed, updates the step size:
        it grows when shifts are longer than expected from random noise and shrinks when they are shorter.

        Args:
            - function, function to optimize.
            - x, y, start point in optimization path.
            - iterations, max iterations before giving up.
            - tolerance, minimum sigma allowed before giving up.
            - popolation size, amount of points to poll.
        Returns:
            Numpy array of (x,y,cost) points.
    """

    #Population properties.
    if population_size is None:
        population_size = cmaes_population_size(n=2)
    elif population_size <= 0:
        raise ValueError("Population size must be positive.")

    #Need non-zero positive tolerance.
    if tolerance<=0:
        raise ValueError("Tolerance must be positive.")

    #Need non-zero positive iterations.
    if iterations<=0:
        raise ValueError("Iterations must be positive count.")

    #Ensure positive sigma.
    if sigma<=0:
        raise ValueError("Sigma must be positive.")
        
    #Best points size.
    elite_size = math.ceil(population_size/2)

    #Weights to place on sorted best points.
    weights = cmaes_weights(population_size, elite_size)

    #Elite size equivalent to equal weighted points.
    effective_size = 1/np.sum(np.asarray(weights)**2)

    #Means logging.
    means = [np.array([x, y], dtype=float)]
    costs = [function(means[-1])]

    #Distribution properties.
    C = np.eye(2)
    mean_path_weight = 2/(3.3**2 + effective_size)
    elite_points_weight = 2*(effective_size - 2 + 1/effective_size)/(4**2 + effective_size)

    #Running averages (trend capture).
    mean_ema = np.zeros(2)
    mean_decay = (4 + effective_size/2)/(6 + effective_size)
    sigma_ema = np.zeros(2)
    sigma_decay = (effective_size + 2)/(effective_size + 7)
    sigma_damping = 1 + 2*max(0, ((effective_size-1)/3)**0.5 - 1) + sigma_decay
    expected_noise_length = (2**0.5)*gamma(1.5)/gamma(1.0)

    for i in range(iterations):
        #Create distribution.
        covariance = sigma**2 * C
        distribution = multivariate_normal(mean=means[-1], cov=covariance, allow_singular=True)

        #Generate and get elite points.
        generated_points = [distribution.rvs() for i in range(population_size)]
        elite_points = sorted(generated_points, key=lambda point: function(point))[:elite_size]

        #Best points mean.
        mean = np.asarray(weights)@np.asarray(elite_points)

        #Normalized mean change.
        normalized_mean_shift = (mean-means[-1])/sigma

        #Update running averages.
        inv_sqrt_cov = np.real(fractional_matrix_power(C,-0.5))
        sigma_ema = (1-sigma_decay)*sigma_ema + ((sigma_decay*(2-sigma_decay)*effective_size)**0.5) * inv_sqrt_cov @ normalized_mean_shift
        mean_ema = (1-mean_decay)*mean_ema + ((mean_decay*(2-mean_decay)*effective_size)**0.5) * normalized_mean_shift
 
        #Update C, covariance, using mean ema trend and elite points differences.
        elites_mean_diffs = np.asarray(elite_points) - means[-1]
        elite_delta_mean_covariance = sum(w*np.outer(d, d) for w, d in zip(weights, elites_mean_diffs))
        C = (1-mean_path_weight-elite_points_weight)*C + mean_path_weight*np.outer(mean_ema, mean_ema) + elite_points_weight/sigma**2 * elite_delta_mean_covariance

        #Ensure symmetry.
        C = (C + C.T)/2 

        #Update sigma using mean.
        sigma *= np.exp(sigma_decay/sigma_damping * (norm(sigma_ema)/expected_noise_length - 1))

        #Break if sigma falls under tolerance.
        if sigma<=tolerance:
            break

        #Log best iteration mean.
        means.append(mean)
        costs.append(function(mean))

    return np.array([(x, y, cost) for (x, y), cost in zip(means, costs)])

In [ ]:
plot_function(cross_in_tray, CMAES(cross_in_tray,2,4, population_size=40))
plot_function(eggholder, CMAES(eggholder,-250,350, population_size=40, sigma=0.1),start_x=-512,end_x=512,start_y=-512,end_y=512)
plot_function(ackley, CMAES(ackley,-8,5,sigma=2, population_size=40))